
# Cálculo Numérico — Estabilidade e Overflow

**Atividade prática II**

Este notebook investiga a relação entre:

- estabilidade numérica;
- passo de tempo $\Delta t$;
- refinamento da malha;
- overflow em `float32` e `float64`.

O problema simula a difusão de velocidade de um fluido entre duas placas paralelas. A placa em $y=0$ se move com velocidade $U=1\ \mathrm{m/s}$ e a placa em $y=H$ permanece parada.



## 1. Verificação da estabilidade

Dados do problema:

$$
H = 0{,}01\ \mathrm{m}
$$

$$
\nu = 10^{-4}\ \mathrm{m^2/s}
$$

$$
N = 21
$$

O espaçamento da malha é:

$$
\Delta y = \frac{H}{N-1}
= \frac{0{,}01}{20}
= 0{,}0005\ \mathrm{m}
$$

Para o método de Euler explícito com diferenças finitas, define-se:

$$
r = \frac{\nu \Delta t}{\Delta y^2}
$$

e a condição de estabilidade é:

$$
r \leq \frac{1}{2}
$$

Assim,

$$
\Delta t_{\max} = \frac{\Delta y^2}{2\nu}
= \frac{(0{,}0005)^2}{2 \cdot 10^{-4}}
= 0{,}00125\ \mathrm{s}
$$

### Caso A

Para $\Delta t=0{,}001$ s:

$$
r = 0{,}4
$$

Como $0{,}4 \leq 0{,}5$, o caso é **estável**.

### Caso B

Para $\Delta t=0{,}002$ s:

$$
r = 0{,}8
$$

Como $0{,}8 > 0{,}5$, o caso é **instável**.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

H = 0.01
nu = 1e-4
pontos = 21

dy = H / (pontos - 1)
dt_max = dy**2 / (2 * nu)

print(f"Δy = {dy:.7f} m")
print(f"Δt máximo = {dt_max:.7f} s")

for dt in (0.001, 0.002):
    r = nu * dt / dy**2
    classificacao = "estável" if r <= 0.5 else "instável"
    print(f"dt={dt:.3f} s -> r={r:.1f} -> {classificacao}")



## 2. Implementação da simulação

A atualização numérica utilizada é:

$$
u_i^{n+1}
=
u_i^n
+
r
\left(
u_{i+1}^n
-
2u_i^n
+
u_{i-1}^n
\right)
$$

O código abaixo interrompe a execução quando detecta overflow ou uma operação numérica inválida.


In [ ]:

def simular(dt, tipo=np.float32, passos=2000, pontos=21):
    H = 0.01
    nu = 1e-4

    y = np.linspace(0, H, pontos)
    dy = H / (pontos - 1)

    r = tipo(nu * dt / dy**2)
    dois = tipo(2)

    u = np.zeros(pontos, dtype=tipo)
    u[0] = tipo(1)

    tempos = [0.0]
    maximos = [float(np.max(np.abs(u)))]
    falha = None

    with np.errstate(over="raise", invalid="raise"):
        for passo in range(1, passos + 1):
            try:
                novo = u.copy()

                novo[1:-1] = (
                    u[1:-1]
                    + r * (
                        u[2:]
                        - dois * u[1:-1]
                        + u[:-2]
                    )
                )

            except FloatingPointError as erro:
                falha = passo
                print(
                    f"dt={dt}, {tipo.__name__}: "
                    f"falha no passo {passo}, "
                    f"t={passo * dt:.4f} s - {erro}"
                )
                break

            u = novo

            tempos.append(passo * dt)
            maximos.append(float(np.max(np.abs(u))))

    if falha is None:
        print(
            f"dt={dt}, {tipo.__name__}: "
            f"{passos} passos sem overflow."
        )

    return {
        "y": y,
        "u": u,
        "tempos": tempos,
        "maximos": maximos,
        "r": float(r),
        "falha": falha,
    }



## 3. Testes com `float32` e `float64`

Serão executadas as quatro combinações pedidas:

1. `float32`, `dt = 0.001`
2. `float32`, `dt = 0.002`
3. `float64`, `dt = 0.001`
4. `float64`, `dt = 0.002`


In [ ]:

print("Maior float32:", np.finfo(np.float32).max)
print("Maior float64:", np.finfo(np.float64).max)

resultados = {}

for tipo in (np.float32, np.float64):
    for dt in (0.001, 0.002):
        nome = f"{tipo.__name__}, dt={dt}"
        resultados[nome] = simular(dt, tipo)


In [ ]:

linhas = []

for nome, resultado in resultados.items():
    tipo_nome, dt_texto = nome.split(", dt=")
    dt = float(dt_texto)

    linhas.append({
        "Tipo numérico": tipo_nome,
        "Δt (s)": dt,
        "r": resultado["r"],
        "Overflow": "Sim" if resultado["falha"] is not None else "Não",
        "Iteração da falha": resultado["falha"] if resultado["falha"] is not None else "-"
    })

tabela = pd.DataFrame(linhas)
tabela



### Interpretação dos testes

Os casos com $\Delta t=0{,}001$ s apresentam $r=0{,}4$, portanto respeitam a condição de estabilidade.

Os casos com $\Delta t=0{,}002$ s apresentam $r=0{,}8$, portanto são instáveis.

A utilização de `float64` **não corrige a instabilidade**. Ela apenas adia o overflow, porque `float64` consegue representar números muito maiores que `float32`.

Assim, a causa do problema não é a capacidade do tipo numérico, mas o fato de o método estar sendo executado fora da sua condição de estabilidade.



## 4. Gráfico da maior velocidade em módulo


In [ ]:

plt.figure(figsize=(9, 5))

for nome, resultado in resultados.items():
    plt.semilogy(
        resultado["tempos"],
        resultado["maximos"],
        label=nome
    )

plt.xlabel("Tempo simulado (s)")
plt.ylabel("Máximo de |u| (m/s) - escala logarítmica")
plt.title("Crescimento da velocidade e overflow")
plt.grid(True)
plt.legend()
plt.show()



### Interpretação física do gráfico

Neste problema, a velocidade física deve permanecer no intervalo:

$$
0 \leq u(y,t) \leq 1\ \mathrm{m/s}
$$

Nos casos estáveis, a solução permanece limitada.

No caso instável, os valores passam a oscilar e crescer rapidamente. Velocidades muito maiores que $1\ \mathrm{m/s}$ não representam o comportamento físico esperado.

Esse crescimento é causado pela **instabilidade numérica**. Como os resultados de uma iteração são utilizados na iteração seguinte, os erros são amplificados sucessivamente. Se esse crescimento continuar, os valores eventualmente ultrapassam a faixa representável pelo tipo numérico e ocorre overflow.



## 5. Refinamento da malha para 41 pontos

Agora usamos:

$$
N=41
$$

O novo espaçamento é:

$$
\Delta y
=
\frac{0{,}01}{41-1}
=
0{,}00025\ \mathrm{m}
$$

O novo limite de estabilidade é:

$$
\Delta t_{\max}
=
\frac{(0{,}00025)^2}{2\cdot10^{-4}}
=
0{,}0003125\ \mathrm{s}
$$

Mantendo inicialmente $\Delta t=0{,}001$ s:

$$
r
=
\frac{10^{-4}(0{,}001)}
{(0{,}00025)^2}
=
1{,}6
$$

Como $1{,}6>0{,}5$, essa configuração é instável.

Isso mostra que, ao refinar a malha, também é necessário reduzir o passo de tempo.


In [ ]:

pontos_refinados = 41
dy_refinado = H / (pontos_refinados - 1)
dt_max_refinado = dy_refinado**2 / (2 * nu)

print(f"Novo Δy = {dy_refinado:.8f} m")
print(f"Novo Δt máximo = {dt_max_refinado:.7f} s")

resultado_refinado_instavel = simular(
    dt=0.001,
    tipo=np.float64,
    passos=2000,
    pontos=41
)

print("r =", resultado_refinado_instavel["r"])



Escolhemos agora:

$$
\Delta t = 0{,}00025\ \mathrm{s}
$$

que é menor que $0{,}0003125$ s.

Nesse caso:

$$
r=0{,}4
$$

e a simulação volta a satisfazer a condição de estabilidade.


In [ ]:

resultado_refinado_estavel = simular(
    dt=0.00025,
    tipo=np.float64,
    passos=10000,
    pontos=41
)

print("r =", resultado_refinado_estavel["r"])



## 6. Comparação com o perfil estacionário

A solução estacionária teórica é:

$$
u_{\mathrm{est}}(y)
=
U\left(1-\frac{y}{H}\right)
$$

Para verificar se a simulação estável se aproximou desse regime, comparamos o perfil numérico com o perfil estacionário.

Também calculamos:

$$
E_\infty
=
\max_i
\left|
u_i-u_{\mathrm{est}}(y_i)
\right|
$$

Esse valor representa a maior diferença entre a solução numérica e o perfil estacionário.


In [ ]:

U = 1.0
H = 0.01

y = resultado_refinado_estavel["y"]
u_numerico = resultado_refinado_estavel["u"]

u_estacionario = U * (1 - y / H)

erro_maximo = np.max(np.abs(u_numerico - u_estacionario))

print(f"Erro máximo E∞ = {erro_maximo:.6e}")

plt.figure(figsize=(9, 5))
plt.plot(y, u_numerico, label="Solução numérica")
plt.plot(y, u_estacionario, "--", label="Perfil estacionário")
plt.xlabel("Posição y (m)")
plt.ylabel("Velocidade u (m/s)")
plt.title("Comparação com o perfil estacionário")
plt.grid(True)
plt.legend()
plt.show()



### Discussão do perfil

Foi utilizada uma execução estável com 41 pontos, $\Delta t=0{,}00025$ s e 10000 passos, correspondendo a $2{,}5$ s de tempo simulado.

O perfil numérico fica praticamente sobreposto ao perfil estacionário. O erro máximo obtido é muito pequeno, indicando que o tempo simulado foi suficiente para aproximar o regime estacionário.

É importante observar que a expressão estacionária descreve o comportamento depois que o transiente praticamente desapareceu. Em tempos menores, uma diferença maior entre os dois perfis não significaria necessariamente erro numérico; parte dessa diferença poderia ser consequência de o sistema ainda estar no regime transiente.



## 7. Respostas finais

### 1. Verificação da estabilidade

Para 21 pontos:

$$
\Delta y=0{,}0005\ \mathrm{m}
$$

e:

$$
\Delta t_{\max}=0{,}00125\ \mathrm{s}
$$

Para $\Delta t=0{,}001$ s:

$$
r=0{,}4
$$

portanto o caso é estável.

Para $\Delta t=0{,}002$ s:

$$
r=0{,}8
$$

portanto o caso é instável.

### 2. Detecção de overflow

Os quatro testes mostram que os casos com $r=0{,}4$ não apresentam overflow durante os 2000 passos, enquanto os casos com $r=0{,}8$ apresentam crescimento instável até ocorrer overflow.

O `float64` demora mais para atingir o overflow porque possui uma faixa de representação muito maior.

### 3. Interpretação física

No caso instável, a velocidade calculada cresce para valores muito superiores a $1\ \mathrm{m/s}$. Isso não representa o comportamento físico esperado, pois a velocidade deveria permanecer entre 0 e $U=1\ \mathrm{m/s}$.

O crescimento é consequência da instabilidade do método numérico.

### 4. Capacidade de representação

Trocar `float32` por `float64` não elimina a instabilidade.

A estabilidade depende da relação:

$$
r=\frac{\nu\Delta t}{\Delta y^2}
$$

O `float64` apenas permite que números maiores sejam armazenados antes que ocorra overflow.

### 5. Refinamento da malha

Com 41 pontos:

$$
\Delta y=0{,}00025\ \mathrm{m}
$$

e:

$$
\Delta t_{\max}=0{,}0003125\ \mathrm{s}
$$

Portanto, manter $\Delta t=0{,}001$ s torna o método instável, pois $r=1{,}6$.

Ao utilizar $\Delta t=0{,}00025$ s, obtém-se $r=0{,}4$, e a simulação volta a ser estável.

### 6. Perfil de velocidade

A execução estável e suficientemente longa se aproxima do perfil:

$$
u_{\mathrm{est}}(y)=U\left(1-\frac{y}{H}\right)
$$

A pequena diferença máxima entre a solução numérica e a estacionária indica que o tempo de simulação escolhido foi suficiente para aproximar o regime estacionário.



## Conclusão

A atividade mostrou que a estabilidade do método numérico depende da relação entre o passo de tempo $\Delta t$ e o espaçamento da malha $\Delta y$.

Para o esquema utilizado, é necessário que:

$$
r=\frac{\nu\Delta t}{\Delta y^2}\leq0{,}5
$$

Quando essa condição não é satisfeita, as oscilações e os erros numéricos podem ser amplificados a cada iteração. Como resultado, surgem velocidades não físicas que crescem continuamente e podem ultrapassar a faixa de representação do tipo numérico, causando overflow.

A utilização de `float64` não corrige a instabilidade; apenas adia o overflow por permitir números de maior magnitude.

Além disso, o refinamento da malha reduz o valor de $\Delta y$ e, consequentemente, reduz também o passo de tempo máximo permitido. Portanto, uma malha mais refinada exige um $\Delta t$ menor para manter a estabilidade.

Nesta simulação, a alteração que realmente corrigiu a causa da falha foi a redução do passo de tempo para um valor que satisfizesse a condição de estabilidade.
